# PC-WMV vs Majority Vote — Paired GSM8K Evaluation on a T4

**Model:** `Qwen/Qwen3-0.6B`  | **Dataset:** GSM8K | **Primary budget:** **16 generated samples/problem**

## Main research question
> **Does PC-WMV improve over standard Majority Vote when both methods use exactly the same model outputs and sampling budget?**

For every GSM8K problem, this notebook runs **one generation pass**:

```text
                    Qwen3-0.6B
                        │
                 generate 16 samples
                        │
          ┌─────────────┼─────────────┐
          ↓             ↓             ↓
         MV        PC-WMV fixed   PC-WMV adaptive
                     τ = 0.75      difficulty-based τ
```

There is **no separate generation run for MV or either PC-WMV variant**. All three aggregators consume the predictions produced by the same experiment run.

### Final comparison
| Method | What it does |
|---|---|
| **MV** | Majority vote over the same generated samples |
| **PC-WMV fixed** | PC-WMV with τ = 0.75 |
| **PC-WMV adaptive** | PC-WMV with the notebook's difficulty-based τ |

Oracle τ is intentionally excluded from the primary comparison.

## 0. T4 setup

Use **Runtime → Change runtime type → T4 GPU**.

This notebook assumes the existing `adaptive-prefix-consistency` project from the uploaded notebook. It keeps the project's PC-WMV implementation and changes the experiment/evaluation protocol around it.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, subprocess, sys, re, ast, math

REPO_DIR = Path('/content/drive/MyDrive/adaptive-prefix-consistency')
if not REPO_DIR.exists():
    raise FileNotFoundError(f"Project not found at {REPO_DIR}. Upload/clone the project there first.")
os.chdir(REPO_DIR)
print('Project:', REPO_DIR)
print('Working directory:', Path.cwd())
!nvidia-smi | head -n 20

Mounted at /content/drive
Project: /content/drive/MyDrive/adaptive-prefix-consistency
Working directory: /content/drive/MyDrive/adaptive-prefix-consistency
Mon Aug 17 03:57:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%

In [2]:
!pip -q install -r requirements.txt
!python scripts/smoke_test.py

All smoke tests passed.


## 1. Experiment configuration

### Primary run
- **Problems:** 1,000
- **Samples/problem:** 16
- **Model:** Qwen3-0.6B
- **One generation pass**
- **Three aggregation outputs:** MV, fixed PC-WMV, adaptive PC-WMV

For a first test, set `N_PROBLEMS = 20`; restore `1000` for the actual experiment.

**Important:** do not run three separate generation commands. Run the experiment once; the resulting rows are used for all three aggregators.

In [5]:
N_PROBLEMS = 100
N_SAMPLES = 16
CONFIG = 'configs/gsm8k_qwen06b.yaml'
OUT = Path('outputs/gsm8k_qwen06b')
print('N_PROBLEMS =', N_PROBLEMS)
print('N_SAMPLES  =', N_SAMPLES)
print('CONFIG     =', CONFIG)

N_PROBLEMS = 100
N_SAMPLES  = 16
CONFIG     = configs/gsm8k_qwen06b.yaml


## 2. Generate once — shared samples for all three methods

This is the **only GPU generation cell**. MV, fixed PC-WMV, and adaptive PC-WMV are evaluated from this same run.

In [6]:
!nvidia-smi

Mon Aug 17 04:48:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import subprocess
import time
import sys

cmd = [
    sys.executable,
    "-u",
    "-m",
    "src.run_experiment",
    "--config",
    "configs/gsm8k_qwen06b.yaml",
    "--num-problems",
    str(N_PROBLEMS),
    "--n-samples",
    "16",
]

print("Running PC-WMV vs MV experiment...", flush=True)
print("Command:", " ".join(cmd), flush=True)

start_time = time.time()

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in iter(process.stdout.readline, ""):
        if line:
            print(line, end="", flush=True)

    process.stdout.close()

    return_code = process.wait()

except KeyboardInterrupt:
    print("\nStopping experiment...", flush=True)
    process.terminate()

    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()

    raise

elapsed = time.time() - start_time

if return_code == 0:
    print(
        f"\nExperiment completed successfully. "
        f"Total runtime: {elapsed / 60:.2f} minutes.",
        flush=True
    )
else:
    print(
        f"\nExperiment failed with return code {return_code}. "
        f"Runtime: {elapsed / 60:.2f} minutes.",
        flush=True
    )

if return_code != 0:
    raise subprocess.CalledProcessError(
        return_code,
        cmd
    )

Streaming output truncated to the last 5000 lines.
      [initial] 15/16 done | tokens=90 | answer='10'
      [initial] sample 16/16
      [initial] 16/16 done | tokens=74 | answer='30'
    [difficulty] agreement=1.000 entropy=0.000 label=easy adaptive_tau=0.450
    [regen] tau=0.750 | 16 traces | k=1
      [regen tau=0.750] 1/16 | answer='30' | tokens=68
      [regen tau=0.750] 2/16 | answer='40' | tokens=109
      [regen tau=0.750] 3/16 | answer='50' | tokens=197
      [regen tau=0.750] 4/16 | answer='10' | tokens=290
      [regen tau=0.750] 5/16 | answer='10' | tokens=309
      [regen tau=0.750] 6/16 | answer='10' | tokens=363
      [regen tau=0.750] 7/16 | answer='30' | tokens=465
      [regen tau=0.750] 8/16 | answer='10' | tokens=530
      [regen tau=0.750] 9/16 | answer='10' | tokens=609
      [regen tau=0.750] 10/16 | answer='10' | tokens=628
      [regen tau=0.750] 11/16 | answer='30' | tokens=675
      [regen tau=0.750] 12/16 | answer='30' | tokens=709
      [regen tau=0.750]

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
import os

path = "outputs/gsm8k_qwen06b_final/results.jsonl"

if os.path.exists(path):
    with open(path, "r", encoding="utf-8") as f:
        print("Completed problems:", sum(1 for _ in f))
else:
    print("Checkpoint not found")

Checkpoint not found


## 3. Load results and verify the paired comparison

Required prediction columns are `mv_pred`, `pc_fixed_pred`, and `pc_adaptive_pred`. Each row is one GSM8K problem, so the three decisions are paired.

In [ ]:
import pandas as pd
if not (OUT / 'results.csv').exists():
    raise FileNotFoundError(f'Missing {OUT / "results.csv"}')
df = pd.read_csv(OUT / 'results.csv')
required = ['gold', 'mv_pred', 'pc_fixed_pred', 'pc_adaptive_pred']
missing = [c for c in required if c not in df.columns]
if missing:
    raise RuntimeError(f'Missing required columns: {missing}')
print(f'Loaded {len(df):,} problems.')
print('PAIRING CHECK: PASS — all three methods have a prediction for every evaluated problem.')
for c in ['n_samples','num_samples','fixed_tau','adaptive_tau','difficulty_label']:
    if c in df.columns:
        print(f'\n{c}:')
        print(df[c].value_counts(dropna=False).head(20))

## 4. Exact three-method evaluation

No Oracle τ in the headline result.

In [ ]:
import numpy as np

def norm_answer(x):
    if pd.isna(x): return ''
    s = str(x).replace(',', '').replace('$', '').strip()
    nums = re.findall(r'-?\d+(?:\.\d+)?', s)
    return nums[-1] if nums else s.lower()

def is_correct(pred, gold):
    p, g = norm_answer(pred), norm_answer(gold)
    return bool(p) and p == g

METHODS = {'MV':'mv_pred', 'PC-WMV fixed':'pc_fixed_pred', 'PC-WMV adaptive':'pc_adaptive_pred'}
correct = {name: np.array([is_correct(p,g) for p,g in zip(df[col],df['gold'])], dtype=bool)
           for name,col in METHODS.items()}
summary = pd.DataFrame({'Method':list(METHODS), 'Correct':[int(correct[m].sum()) for m in METHODS],
                        'Total':[len(df)]*3, 'Accuracy':[correct[m].mean() for m in METHODS]})
summary['Accuracy (%)'] = 100*summary['Accuracy']
mv_acc = summary.loc[summary.Method=='MV','Accuracy'].iloc[0]
summary['Delta vs MV (pp)'] = 100*(summary['Accuracy']-mv_acc)
display(summary[['Method','Correct','Total','Accuracy (%)','Delta vs MV (pp)']].round(3))

## 5. Paired win/loss analysis

In [ ]:
def paired_counts(a,b):
    return {'both_correct':int((a&b).sum()), 'a_only_correct':int((a&~b).sum()),
            'b_only_correct':int((~a&b).sum()), 'both_wrong':int((~a&~b).sum())}
comparisons=[('MV','PC-WMV fixed'),('MV','PC-WMV adaptive'),('PC-WMV fixed','PC-WMV adaptive')]
rows=[]
for a,b in comparisons:
    x=paired_counts(correct[a],correct[b])
    rows.append({'Comparison':f'{a} vs {b}', **x})
paired_df=pd.DataFrame(rows)
display(paired_df)

## 6. McNemar exact paired test

In [ ]:
from scipy.stats import binomtest

def mcnemar_exact(a,b):
    a_only=int((a&~b).sum()); b_only=int((~a&b).sum()); n=a_only+b_only
    p=1.0 if n==0 else float(binomtest(b_only,n=n,p=0.5,alternative='two-sided').pvalue)
    return a_only,b_only,p
rows=[]
for a,b in comparisons:
    ao,bo,p=mcnemar_exact(correct[a],correct[b])
    rows.append({'A':a,'B':b,'A-only wins':ao,'B-only wins':bo,'McNemar exact p':p})
mcnemar_df=pd.DataFrame(rows)
display(mcnemar_df)

## 7. Paired bootstrap 95% confidence intervals

In [ ]:
RNG=np.random.default_rng(42); B=5000

def boot_ci(x):
    n=len(x); idx=RNG.integers(0,n,size=(B,n)); vals=x[idx].mean(axis=1)
    return np.quantile(vals,[.025,.975])

def boot_delta_ci(a,b):
    n=len(a); idx=RNG.integers(0,n,size=(B,n)); vals=b[idx].mean(axis=1)-a[idx].mean(axis=1)
    return np.quantile(vals,[.025,.975])
rows=[]
for name in METHODS:
    lo,hi=boot_ci(correct[name]); rows.append({'Method':name,'Accuracy (%)':100*correct[name].mean(),
        '95% CI low (%)':100*lo,'95% CI high (%)':100*hi})
ci_df=pd.DataFrame(rows); display(ci_df.round(3))
for name in ['PC-WMV fixed','PC-WMV adaptive']:
    lo,hi=boot_delta_ci(correct['MV'],correct[name])
    print(f'{name} - MV: {100*(correct[name].mean()-correct["MV"].mean()):.3f} pp; 95% paired bootstrap CI = [{100*lo:.3f}, {100*hi:.3f}] pp')

## 8. Difficulty-stratified analysis

In [ ]:
if 'difficulty_label' in df.columns:
    rows=[]
    for label in ['easy','medium','hard']:
        mask=df['difficulty_label'].astype(str).str.lower().eq(label)
        if mask.any():
            rows.append({'Difficulty':label,'N':int(mask.sum()),
                         'MV (%)':100*correct['MV'][mask].mean(),
                         'PC-WMV fixed (%)':100*correct['PC-WMV fixed'][mask].mean(),
                         'PC-WMV adaptive (%)':100*correct['PC-WMV adaptive'][mask].mean()})
    difficulty_df=pd.DataFrame(rows); display(difficulty_df.round(2))
else:
    print('difficulty_label not present; skipping.')

## 9. Adaptive τ audit

In [ ]:
if 'adaptive_tau' in df.columns:
    display(df['adaptive_tau'].value_counts().sort_index().rename('count').to_frame())
    if 'fixed_tau' in df.columns:
        print('Adaptive τ differs from fixed τ on:', int((df['adaptive_tau']!=df['fixed_tau']).sum()), '/', len(df), 'problems')
else:
    print('adaptive_tau column not present.')

## 10. Paper-ready plots — only the three primary methods

In [ ]:
import matplotlib.pyplot as plt
PLOT=OUT/'paired_paper_plots'; PLOT.mkdir(parents=True,exist_ok=True)
colors=['#222222','#2F6F8F','#C46A3C']
fig,ax=plt.subplots(figsize=(7,4.3)); vals=summary['Accuracy (%)'].to_numpy(); labels=summary['Method'].tolist()
bars=ax.bar(labels,vals,color=colors,width=.68); ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0,max(vals)+10)
ax.set_title('GSM8K — Qwen3-0.6B — matched 16-sample aggregation'); ax.tick_params(axis='x',rotation=12)
for b,v in zip(bars,vals): ax.text(b.get_x()+b.get_width()/2,v+.6,f'{v:.2f}',ha='center')
fig.tight_layout(); fig.savefig(PLOT/'accuracy_three_methods.png',dpi=300,bbox_inches='tight'); plt.show()

fig,ax=plt.subplots(figsize=(7,3.6)); ax.axis('off')
rows=[]
for a,b in comparisons:
    x=paired_counts(correct[a],correct[b]); rows.append([f'{a} vs {b}',x['both_correct'],x['a_only_correct'],x['b_only_correct'],x['both_wrong']])
t=ax.table(cellText=rows,colLabels=['Comparison','Both correct','A only','B only','Both wrong'],loc='center',cellLoc='center'); t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1,1.7)
ax.set_title('Paired outcomes on the same GSM8K problems',pad=16); fig.tight_layout(); fig.savefig(PLOT/'paired_outcomes.png',dpi=300,bbox_inches='tight'); plt.show()
print('Plots saved to:',PLOT.resolve())

## 11. Save final three-method outputs

In [ ]:
summary.to_csv(PLOT/'three_method_summary.csv',index=False)
paired_df.to_csv(PLOT/'paired_outcomes.csv',index=False)
mcnemar_df.to_csv(PLOT/'mcnemar_tests.csv',index=False)
ci_df.to_csv(PLOT/'bootstrap_accuracy_ci.csv',index=False)
if 'difficulty_df' in globals(): difficulty_df.to_csv(PLOT/'difficulty_analysis.csv',index=False)
print('Saved outputs to',PLOT.resolve())

## 12. Interpretation

The clean comparison is:

> **At a matched budget of 16 Qwen3-0.6B samples per GSM8K problem, PC-WMV is compared directly against Majority Vote using exactly the same generated outputs; any accuracy difference is attributable to the aggregation rule rather than different model generations.**

Do not claim improvement until the paired results actually show it. Oracle τ is excluded from the headline result.